In [0]:
import pkg_resources

In [0]:
%pip list

In [0]:
!pip install -U --quiet PyMUPDF langchain databricks-langchain

In [0]:
%pip install --upgrade typing_extensions

In [0]:
%restart_python

In [0]:
file_path = "/Volumes/workspace/udemy/demovol/dpact.pdf"

import fitz  # PyMuPDF

# Read file content
with open(file_path, "rb") as f:
    pdf_bytes = f.read()

# Use PyMuPDF to extract text
doc = fitz.open("pdf", pdf_bytes)

text = ""
for page in doc:
    text += page.get_text()

print(text)  # Print first 1000 characters

# CSV Example
df = spark.read.option("header", "true").csv("/Volumes/<catalog>/<schema>/<table>")

# OR Parquet
# df = spark.read.parquet("/Volumes/<catalog>/<schema>/<table>")

# OR JSON
# df = spark.read.option("multiline", "true").json("/Volumes/<catalog>/<schema>/<table>")

In [0]:
%pip list | grep databricks-langchain

In [0]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os
import pandas as pd

# Load documents (e.g., text files or PDFs from DBFS or local)
raw_text = text
# Replace this with your actual file reader logic

# Chunk documents for embedding
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
docs = text_splitter.create_documents([raw_text])

# Convert docs to a list of dicts for display
pd_docs = pd.DataFrame([doc.dict() for doc in docs])

# Add an index column
pd_docs.insert(0, "id_pk", range(1, len(pd_docs) + 1))

display(pd_docs)

In [0]:
len(docs)

In [0]:
docs[1]

In [0]:
# Create Spark DataFrame from pandas DataFrame
spark_df = spark.createDataFrame(pd_docs[['id_pk', 'page_content']])

# Write to Delta table
spark_df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("workspace.udemy.my_data_chunks")

In [0]:
# Import and initialize Vector Search Client
from databricks.vector_search.client import VectorSearchClient

client = VectorSearchClient()

# [NOTICE] Using a notebook authentication token.
# Recommended for development only.
# For improved performance, use Service-based authentication.
# To disable this message, pass disable_notice=True.

In [0]:
def endpoint_exists__old1(vsc, endpoint_name):
    endpoints = vsc.list_endpoints()
    if isinstance(endpoints, str):
        raise ValueError("Expected a list of endpoints, but got a string.")
    return any(ep['name'] == endpoint_name for ep in endpoints)

In [0]:
def endpoint_exists(vsc, endpoint_name):
    endpoints_response = vsc.list_endpoints()
    # If endpoints_response is a dict, extract the list
    if isinstance(endpoints_response, dict) and "endpoints" in endpoints_response:
        endpoints = endpoints_response["endpoints"]
    elif isinstance(endpoints_response, list):
        endpoints = endpoints_response
    else:
        raise ValueError(
            "Expected a list or dict with 'endpoints' key, but got: {}".format(type(endpoints_response))
        )
    return any(ep['name'] == endpoint_name for ep in endpoints)

In [0]:
import time

def wait_for_index_to_be_ready(vsc, endpoint_name,index_name, timeout=300):
    start_time = time.time()
    while time.time() - start_time < timeout:
        index_info = vsc.get_index(endpoint_name, index_name)
        if index_info['status'] == 'READY':
            return True
        time.sleep(5)  # Wait before checking again
    raise TimeoutError(f"Index {index_name} did not become ready in {timeout} seconds.")

In [0]:
VECTOR_SEARCH_ENDPOINT_NAME ="vs_endpoint1"
if not endpoint_exists(client, VECTOR_SEARCH_ENDPOINT_NAME):
    client.create_endpoint(name=VECTOR_SEARCH_ENDPOINT_NAME, endpoint_type="STANDARD")

print(f"Endpoint named {VECTOR_SEARCH_ENDPOINT_NAME} is ready.")

In [0]:
%sql
ALTER TABLE workspace.udemy.my_data_chunks
SET TBLPROPERTIES (delta.enableChangeDataFeed = true)

In [0]:
print(client.list_endpoints())

In [0]:
def index_exists(
    vsc,
    endpoint_name,
    index_name
):
    indexes_response = vsc.list_indexes(endpoint_name )
    # If indexes_response is a dict and has 'indexes', extract the list
    if isinstance(indexes_response, dict) and "indexes" in indexes_response:
        indexes = indexes_response["indexes"]
    # If indexes_response is a dict but does not have 'indexes', treat as list of indexes
    elif isinstance(indexes_response, dict):
        indexes = [indexes_response]
    elif isinstance(indexes_response, list):
        indexes = indexes_response
    else:
        raise ValueError(
            f"Expected a list or dict with 'indexes' key, but got: {type(indexes_response)}"
        )
    print(indexes)
    flag = False
    for idx in indexes:
        if(idx.get('name') == index_name):
            flag = True
            print("Index name:", idx.get('name'))
            break           
    print("Exists:", flag)
    #flag = any(idx.get('name') == index_name for idx in indexes )
    #print(flag)
    #return any(idx.get('name') == index_name for idx in indexes )
    return flag

In [0]:
# creates Vector Search index named documents_index, and sets - Vector Search endpoint
index_name="workspace.udemy.my_data_chunks_index"
if (index_exists(client, VECTOR_SEARCH_ENDPOINT_NAME, index_name)):
  print(f"Creating index {index_name} on endpoint {VECTOR_SEARCH_ENDPOINT_NAME}...")
  vs_index = client.create_delta_sync_index(
        endpoint_name="vs_endpoint1",
        index_name="workspace.udemy.my_data_chunks_index",
        source_table_name="workspace.udemy.my_data_chunks",  
        primary_key="id_pk",
        embedding_source_column="page_content",
        embedding_model_endpoint_name='databricks-gte-large-en',
        pipeline_type="TRIGGERED"
    )
else:
    print(f"Index {index_name} already exists.")
    vs_index = client.get_index(endpoint_name="vs_endpoint1", index_name="workspace.udemy.my_data_chunks_index")


In [0]:
results_dict = vs_index.similarity_search(
    query_text = "Who is Data fudiciary",
    columns=["id_pk","page_content"],
    num_results=2
)
display(results_dict)

In [0]:
input="Answer Question " + str(results_dict) + " Question: Who is data fudiciary"
#print(input)

In [0]:
from databricks_langchain import ChatDatabricks
chatmodel= ChatDatabricks(model="databricks-meta-llama-3-3-70b-instruct",temperature=0, max_tokens=250)
chatmodel.invoke(input)

In [0]:
# there i sno continuety from above
# Now using lanagchain framework how can we include vector search with with model 
# and vector search seemlessly 
from databricks_langchain import DatabricksVectorSearch

vector_store= DatabricksVectorSearch(index_name="workspace.udemy.my_data_chunks_index")
retriever = vector_store.as_retriever(search_kwargs={"k": 2})
relevent_documents =retriever.invoke("What is data fudiciary")

In [0]:
chain_config = {
    "llm_model_serving_endpoint_name": "databricks-meta-llama-3-3-70b-instruct",  # the foundation model we want to use
    "vector_search_endpoint_name": VECTOR_SEARCH_ENDPOINT_NAME,  # the endoint we want to use for vector search
    "vector_search_index": "workspace.udemy.my_data_chunks_index",
    "llm_prompt_template": """You are an assistant that answers questions. Use the following pieces of retrieved context to answer the question. Some pieces of context may be irrelevant, in which case you should not use them to form the answer.\n\nContext: {context}""",
}

In [0]:
# Method to format the docs returned by the retriever into the prompt (keep only the text from chunks)
def format_context(docs):
    chunk_contents = [f"Passage: {d.page_content}\n" for d in docs]
    return "".join(chunk_contents)

In [0]:
format_context(relevent_documents)

In [0]:
from langchain_core.prompts import ChatPromptTemplate
from databricks_langchain.chat_models import ChatDatabricks
from operator import itemgetter

prompt = ChatPromptTemplate.from_messages(
    [  
        ("system", chain_config.get("llm_prompt_template")), # Contains the instructions from the configuration
        ("user", "{question}") #user's questions
    ]
)


In [0]:
prompt

In [0]:
from langchain_core.output_parsers import StrOutputParser
# Our foundation model answering the final prompt
#model = ChatDatabricks(
#    endpoint=model_config.get("llm_model_serving_endpoint_name"),
#    extra_params={"temperature": 0.01, "max_tokens": 500}
#)

#Let's try our prompt:
answer = (prompt | chatmodel | StrOutputParser()).invoke({'question':'who is data fudiciary?', 'context': ''})
print(answer)